# OneData-Omni · 10-scene MOVi-C multisensory smoke test
### Real MOVi-C video + geometry/state + event-aligned *synthetic* tactile/impact audio + spoken QA

**Goal:** Make exactly **10 aligned records** from the Google/Kubric **MOVi-C (128×128)** prepared dataset, then **play and inspect a multimodal demo**. No model training and **no H200 required** (Colab CPU/T4 both work).

**What is native MOVi-C?** RGB video, segmentation, quantized depth, forward optical flow, object-centric 3D positions/velocities/3D boxes, and collision frames/forces. **What we add?** contact-derived tactile *proxy* (not a physical GelSight simulator), synthesized impact WAV (not a realistic acoustic simulator), offline generated spoken English question WAV (scene-level alignment, not in-world simultaneous speech), and approximate camera-coordinate point cloud from RGB-D. **Original full GSO meshes are NOT bundled with MOVi-C TFDS samples**.

**Data provenance is explicit:** `data_origin` will be `MOVi-C TFDS` on success; an optional `mock_selftest` mode exists solely to verify this export notebook without Google Cloud. Mock data is *not* MOVi-C and cannot be presented as such.

Official source: [Kubric MOVi instructions](https://github.com/google-research/kubric/blob/main/challenges/movi/README.md) · [MOVi-C feature definitions](https://github.com/google-research/kubric/blob/main/challenges/movi/movi_c.py).

**Run All** in Colab; the final cell includes an interactive scene picker, MP4, audio, point cloud, tactile, RGB/depth/segmentation and aligned time series.

## 0 · Install lightweight dependencies

TensorFlow is normally preinstalled in Colab. We read the **already prepared public TFDS shards** rather than regenerating the whole Kubric dataset. `tfds.load(..., split="train[:10]", download=False)` asks for ten examples; access still depends on Google Cloud Storage connectivity. Offline speech uses `espeak-ng`, so no paid API is needed.

In [1]:
%pip -q install "tensorflow-datasets>=4.9.6" "imageio[ffmpeg]" "imageio-ffmpeg" soundfile scipy matplotlib ipywidgets
!apt-get -qq update >/dev/null && apt-get -qq install -y espeak-ng >/dev/null
print("Dependencies installed. If TensorFlow import is missing, install tensorflow in this Colab runtime.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Dependencies installed. If TensorFlow import is missing, install tensorflow in this Colab runtime.


## 1 · Configuration and imports

**Default** is real MOVi-C, exactly 10 records. Only switch `DATA_SOURCE = "mock_selftest"` when you need to test the pipeline without public GCS access; mock mode is a separate internal test and has a clearly different provenance tag.

In [2]:
from pathlib import Path
import os, json, math, subprocess, shutil, io, warnings
import numpy as np
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import soundfile as sf
from PIL import Image, ImageDraw
from IPython.display import display, Audio, Video, Markdown

DATA_SOURCE = "movi_c"              # "movi_c" (real) or "mock_selftest" (synthetic test only)
N_SCENES = 10
TFDS_DATASET = "movi_c/128x128"
TFDS_DATA_DIR = "gs://kubric-public/tfds"
SPLIT = f"train[:{N_SCENES}]"
FPS = 12.0
AUDIO_SR = 16000
TOUCH_SIZE = 32
MAX_POINT_SAMPLES = 3200
SEED = 20261120
OUTPUT_ROOT = Path("/content/OneData_Omni_MOViC_10" if Path('/content').exists() else './OneData_Omni_MOViC_10')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Output:", OUTPUT_ROOT.resolve())
print("Source:", DATA_SOURCE, "; count:", N_SCENES)

rng_global = np.random.default_rng(SEED)

def as_python(v):
    if isinstance(v, (bytes, np.bytes_)): return v.decode('utf-8', errors='replace')
    if isinstance(v, np.ndarray): return v.tolist()
    if isinstance(v, (np.integer, np.floating)): return v.item()
    return v

def text_value(x):
    if isinstance(x, np.ndarray) and x.ndim == 0: x = x.item()
    return str(as_python(x))

Output: /content/OneData_Omni_MOViC_10
Source: movi_c ; count: 10


## 2 · Fetch only the first 10 prepared MOVi-C examples

Dataset is hosted by Google as prepared TFDS shards. **We do not call `download_and_prepare`**: that would require generating/downloading the *entire dataset*. If Google Cloud Storage is inaccessible, the cell fails transparently with a diagnostic instead of replacing the data silently.

In [3]:
if DATA_SOURCE == "movi_c":
    try:
        import tensorflow as tf
        import tensorflow_datasets as tfds
        print("TensorFlow", tf.__version__, "TFDS", tfds.__version__)
        ds = tfds.load(TFDS_DATASET,
                       data_dir=TFDS_DATA_DIR,
                       split=SPLIT,
                       download=False,
                       shuffle_files=False)
        records = list(tfds.as_numpy(ds))
        assert len(records) == N_SCENES, f"Expected {N_SCENES} MOVi-C examples, got {len(records)}"
        print(f"REAL MOVi-C loaded: {len(records)} examples")
        print("Fields:", sorted(records[0].keys()))
        print("Instances:", sorted(records[0]['instances'].keys()))
        print("Collisions:", sorted(records[0]['events']['collisions'].keys()))
        DATA_ORIGIN = "MOVi-C TFDS (google-research/kubric)"
    except Exception as e:
        raise RuntimeError(
            "Public MOVi-C TFDS could not be read. Check Google Cloud Storage internet access and "
            "that 'gs://kubric-public/tfds' is reachable from Colab. Try Runtime > Restart session, "
            "or consult the Kubric README. For an exporter-only test, explicitly set "
            "DATA_SOURCE='mock_selftest' in the previous cell and rerun. "
            "A mock test is NOT equivalent to 10 real MOVi-C samples."
        ) from e
else:
    # Structurally compatible, purposely obvious synthetic fixture for testing only.
    def build_mock_record(i):
        T,H,W,N = 24,128,128,3
        rr=np.random.default_rng(i+1234)
        video=np.full((T,H,W,3), [220,231,243], np.uint8)
        seg=np.zeros((T,H,W,1), np.uint8)
        depth=np.zeros((T,H,W,1), np.uint16)
        positions=np.zeros((N,T,3),np.float32)
        velocities=np.zeros_like(positions)
        bboxes=np.zeros((N,T,8,3),np.float32)
        for t in range(T):
            for j in range(N):
                x=18+j*38+int((t-10)*(j+1)*.3)
                y=30+j*22+int(7*np.sin(t/5+j))
                x=int(np.clip(x,5,W-12)); y=int(np.clip(y,6,H-12))
                color=([234,82,78],[60,133,238],[64,194,133])[j]
                frame=Image.fromarray(video[t]);d=ImageDraw.Draw(frame)
                d.ellipse([x-9,y-9,x+9,y+9], fill=tuple(color))
                video[t]=np.asarray(frame)
                Y,X=np.ogrid[:H,:W]; m=(X-x)**2+(Y-y)**2<=9**2
                seg[t,m,0]=j+1; depth[t,m,0]=np.uint16((1+j*.18)*12000)
                positions[j,t]=[x/40.,y/40.,j*.1]
                bboxes[j,t]=positions[j,t][None]+rr.uniform(-.15,.15,(8,3))
        velocities=np.gradient(positions,1/FPS,axis=1).astype(np.float32)
        event_frame=np.array([7,13,19],np.int32)
        return dict(video=video,depth=depth,segmentations=seg,
            metadata=dict(video_name=f"mock_{i:04d}".encode(), depth_range=np.array([0.,4.],np.float32),
                          num_frames=np.int32(T), num_instances=np.uint16(N)),
            instances=dict(positions=positions,velocities=velocities,angular_velocities=np.zeros_like(velocities),
                bboxes_3d=bboxes,asset_id=np.array([f"MOCK{i}_{j}".encode() for j in range(N)]),
                category=np.arange(N,dtype=np.uint8),visibility=np.ones((N,T),np.float32)),
            camera=dict(field_of_view=np.float32(.85756),positions=np.zeros((T,3),np.float32)),
            events=dict(collisions=dict(frame=event_frame,force=np.array([15.,40.,24.],np.float32),
                instances=np.array([[0,65535],[0,1],[1,2]],np.uint16),
                position=np.zeros((3,3),np.float32),contact_normal=np.tile([0.,0.,1.],(3,1)).astype(np.float32),
                image_position=np.array([[.25,.3],[.5,.5],[.7,.65]],np.float32))))
    records=[build_mock_record(i) for i in range(N_SCENES)]
    DATA_ORIGIN="MOCK SELF-TEST — NOT MOVi-C"
    print("WARNING:",DATA_ORIGIN,"; no claims of real MOVi-C data are valid.")

TensorFlow 2.21.0 TFDS 4.9.10


ERROR:grpc._plugin_wrapping:AuthMetadataPluginCallback "<google.auth.transport.grpc.AuthMetadataPlugin object at 0x7b4ec0a6d2b0>" raised exception!
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/google/auth/compute_engine/credentials.py", line 139, in _perform_refresh_token
    self._retrieve_info(request)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/google/auth/compute_engine/credentials.py", line 107, in _retrieve_info
    info = _metadata.get_service_account_info(
        request, service_account=self._service_account_email
    )
  File "/usr/local/lib/python3.13/dist-packages/google/auth/compute_engine/_metadata.py", line 461, in get_service_account_info
    return get(request, path, params={"recursive": "true"})
  File "/usr/local/lib/python3.13/dist-packages/google/auth/compute_engine/_metadata.py", line 386, in get
    raise exceptions.TransportError(
    ...<5 lines>...
    )
google.auth.exceptions.Transpor

REAL MOVi-C loaded: 10 examples
Fields: ['background', 'backward_flow', 'camera', 'depth', 'events', 'forward_flow', 'instances', 'metadata', 'normal', 'object_coordinates', 'segmentations', 'video']
Instances: ['angular_velocities', 'asset_id', 'bbox_frames', 'bboxes', 'bboxes_3d', 'category', 'friction', 'image_positions', 'mass', 'positions', 'quaternions', 'restitution', 'scale', 'velocities', 'visibility']
Collisions: ['contact_normal', 'force', 'frame', 'image_position', 'instances', 'position']


## 3 · Exporters: real modality values and explicitly synthetic proxies

- **Depth:** TFDS stores quantized uint16. Reconstruct using each record's `depth_range`.
- **3D:** Object positions, velocities and 3D bounding boxes are native. The point cloud is a **backprojection of frame-0 RGB-D**, in camera coordinates; it is *not* the original GSO mesh.
- **Tactile:** For each annotated collision, place a smooth pressure patch whose strength depends on annotated force. This is a **contact signal proxy**, **not** physical tactile rendering.
- **Impact audio:** Generated from collision frames using decaying noise/tones. Not acoustically realistic. It *does* share the same discrete event timestamps.
- **Speech:** A spoken QA prompt is linked to the scene, not pretending to be an event-synchronized microphone recording.

In [4]:
def denormalize_depth(u16, depth_range):
    u=np.asarray(u16, dtype=np.float32)
    lo,hi=np.asarray(depth_range,dtype=np.float32).reshape(-1)[:2]
    return (u/65535.0*(hi-lo)+lo).astype(np.float32)

def backproject_first_frame(rgb, depth, seg, fov, limit=3200, seed=0):
    H,W=depth.shape
    fov=float(fov)
    if fov>np.pi: fov=np.deg2rad(fov)
    fx=(W/2)/np.tan(fov/2); fy=fx
    yy,xx=np.indices((H,W))
    # Exclude background: this gives an object-only RGB-D point sample.
    valid=(seg>0)&np.isfinite(depth)&(depth>1e-5)
    ids=np.flatnonzero(valid.reshape(-1))
    if ids.size==0:
        valid=np.isfinite(depth)&(depth>1e-5)
        ids=np.flatnonzero(valid.reshape(-1))
    if ids.size==0:
        return np.zeros((0,3),np.float32),np.zeros((0,3),np.uint8),np.zeros((0,),np.uint8)
    rng=np.random.default_rng(seed)
    ids=rng.choice(ids,size=min(limit,ids.size),replace=False)
    y,x=np.unravel_index(ids,(H,W))
    ray=np.stack([(x-(W-1)/2)/fx,-(y-(H-1)/2)/fy,-np.ones_like(x)],axis=1).astype(np.float32)
    ray=ray/np.linalg.norm(ray,axis=1,keepdims=True)
    # IMPORTANT: MOVi depth is radial distance from camera center (not z-depth).
    p=ray*depth[y,x,None]
    return p.astype(np.float32),rgb[y,x].astype(np.uint8),seg[y,x].astype(np.uint8)

def collision_dict(events):
    raw=events.get('collisions',{})
    frame=np.asarray(raw.get('frame',np.empty((0,),np.int32))).reshape(-1).astype(np.int32)
    n=len(frame)
    def shaped(name,tail,typ):
        arr=np.asarray(raw.get(name,np.empty((n,*tail))),dtype=typ)
        return arr.reshape(n,*tail)
    return dict(frame=frame,force=shaped('force',(),np.float32).reshape(-1),
        instances=shaped('instances',(2,),np.int32),
        position=shaped('position',(3,),np.float32),
        image_position=shaped('image_position',(2,),np.float32),
        contact_normal=shaped('contact_normal',(3,),np.float32))

def synthesize_contact_proxies(coll, n_frames, fps=12., sr=16000, grid=32, seed=0):
    force=np.zeros((n_frames,),np.float32)
    tactile=np.zeros((n_frames,grid,grid),np.float32)
    length=int(round(n_frames/fps*sr))
    audio=np.zeros((length,),np.float32)
    rr=np.random.default_rng(seed)
    yy,xx=np.mgrid[:grid,:grid]
    fframes=coll['frame']; mags=np.maximum(0,np.asarray(coll['force'],np.float32))
    valid=(fframes>=0)&(fframes<n_frames)&np.isfinite(mags)&(mags>0)
    scale=float(np.max(np.log1p(mags[valid]))) if valid.any() else 1.
    # Aggregate force per video frame, and synthesize the same timestamped event in tactile and WAV.
    for k in np.flatnonzero(valid):
        frame=int(fframes[k]); mag=float(mags[k]); force[frame]+=mag
        amp=float(np.log1p(mag)/max(scale,1e-6))
        normal=coll['contact_normal'][k]
        # Schematic contact patch, local to an imaginary tactile pad, not a calibrated sensor.
        cx=(grid-1)/2+float(np.clip(normal[0],-1,1))*grid*.17
        cy=(grid-1)/2-float(np.clip(normal[1],-1,1))*grid*.17
        sigma=max(grid*.09,2.)
        tactile[frame]+=amp*np.exp(-((xx-cx)**2+(yy-cy)**2)/(2*sigma*sigma))
        start=int(round(frame/fps*sr))
        count=min(int(.095*sr),len(audio)-start)
        if count<=0: continue
        tt=np.arange(count,dtype=np.float32)/sr
        freq=350+85*(int(coll['instances'][k][0])%7)+40*(int(coll['instances'][k][1])%5)
        wave=np.exp(-tt*34.)*(np.sin(2*np.pi*freq*tt)+.23*rr.standard_normal(count))
        audio[start:start+count]+=amp*wave.astype(np.float32)*.20
    if np.max(np.abs(audio))>0.96: audio=audio/np.max(np.abs(audio))*.96
    return force,tactile,audio

def create_qa(coll,n_frames,fps):
    valid=(coll['frame']>=0)&(coll['frame']<n_frames)&(coll['force']>0)
    inds=np.flatnonzero(valid)
    if not len(inds):
        return dict(question="Was any collision recorded in this video?",answer="No collision was recorded.",
                    event_frame=None,object_ids=[], event_type="no_collision")
    i=int(inds[np.argmax(coll['force'][inds])])
    a,b=[int(v) for v in coll['instances'][i]]
    sec=float(coll['frame'][i]/fps)
    question="Which objects were involved in the strongest recorded collision, and when did it happen?"
    if b==65535 or a==65535:
        idx=a if b==65535 else b
        answer=f"Object {idx} collided with the floor at about {sec:.2f} seconds."
    else:
        answer=f"Objects {a} and {b} collided at about {sec:.2f} seconds."
    return dict(question=question,answer=answer,event_frame=int(coll['frame'][i]),
                event_time_s=sec,object_ids=[a,b],event_type="collision")

def speak_offline(question,wav_path):
    exe=shutil.which('espeak-ng') or shutil.which('espeak')
    if exe is None: raise EnvironmentError("espeak-ng not installed. Re-run install cell.")
    subprocess.run([exe,'-v','en-us','-s','140','-w',str(wav_path),str(question)],check=True,
                   stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
    return str(wav_path)

def save_video(arr,filepath,fps=12.):
    arr=np.asarray(arr)
    assert arr.dtype==np.uint8 and arr.ndim==4 and arr.shape[-1]==3
    imageio.mimsave(str(filepath),arr,fps=fps,macro_block_size=1,codec='libx264',quality=7)

def array_json(value):
    if isinstance(value,dict): return {str(k):array_json(v) for k,v in value.items()}
    if isinstance(value,(list,tuple)): return [array_json(v) for v in value]
    if isinstance(value,np.ndarray): return value.tolist()
    if isinstance(value,(np.integer,np.floating)): return value.item()
    if isinstance(value,(bytes,np.bytes_)): return value.decode('utf-8',errors='replace')
    return value

## 4 · Generate exactly 10 aligned records

Per scene save MP4, array NPZ, object states and collision table, point cloud, contact proxy, synthesized impact WAV, spoken question WAV, QA and manifest. Collision events are direct from Kubric; *no colliding events are invented* for real MOVi-C samples.

In [5]:
def export_record(ex,scene_index,root=OUTPUT_ROOT,origin=DATA_ORIGIN):
    scene_id=f"scene_{scene_index:04d}"
    out=root/scene_id;out.mkdir(parents=True,exist_ok=True)
    rgb=np.asarray(ex['video'],dtype=np.uint8)
    T,H,W,_=rgb.shape
    seg=np.asarray(ex['segmentations'],dtype=np.uint8).reshape(T,H,W)
    depth=denormalize_depth(np.asarray(ex['depth']).reshape(T,H,W),ex['metadata']['depth_range'])
    inst=ex['instances']
    pos=np.asarray(inst['positions'],np.float32)
    vel=np.asarray(inst['velocities'],np.float32)
    assert pos.shape[1:]==(T,3) and vel.shape==pos.shape, (pos.shape,vel.shape,T)
    nobj=pos.shape[0]
    coll=collision_dict(ex['events'])
    force,tactile,impact=synthesize_contact_proxies(coll,T,FPS,AUDIO_SR,TOUCH_SIZE,SEED+scene_index)
    qa=create_qa(coll,T,FPS)
    # Preserve native RGB/depth/segmentation and flow for future representation experiments.
    native=dict(rgb=rgb,depth_camera_distance=depth,segmentation=seg,
                depth_range_original=np.asarray(ex['metadata']['depth_range'],np.float32))
    if 'forward_flow' in ex: native['forward_flow_uint16']=np.asarray(ex['forward_flow'],np.uint16)
    if 'normal' in ex: native['normal_uint16']=np.asarray(ex['normal'],np.uint16)
    if 'object_coordinates' in ex: native['object_coordinates_uint16']=np.asarray(ex['object_coordinates'],np.uint16)
    np.savez_compressed(out/'visual.npz',**native)
    np.savez_compressed(out/'motion.npz',positions_world=pos,velocities_world=vel,
        angular_velocities_world=np.asarray(inst.get('angular_velocities',np.zeros_like(vel)),np.float32),
        bboxes_3d_world=np.asarray(inst.get('bboxes_3d',np.empty((0,))),np.float32),
        frame_timestamp_s=np.arange(T,dtype=np.float32)/FPS)
    np.savez_compressed(out/'collisions.npz',**coll)
    np.savez_compressed(out/'contact_tactile_proxy.npz',force_by_frame=force,tactile_contact_patch=tactile,
                        frame_timestamp_s=np.arange(T,dtype=np.float32)/FPS)
    save_video(rgb,out/'rgb.mp4',fps=FPS)
    sf.write(out/'impact_synthetic.wav',impact,AUDIO_SR)
    speak_offline(qa['question'],out/'spoken_question.wav')
    p,prgb,pids=backproject_first_frame(rgb[0],depth[0],seg[0],
        ex['camera'].get('field_of_view',.85756),MAX_POINT_SAMPLES,SEED+scene_index)
    np.savez_compressed(out/'pointcloud_camera_frame.npz',xyz=p,rgb=prgb,instance_mask_id=pids)
    with open(out/'qa.json','w') as f: json.dump(qa,f,indent=2)
    assets=inst.get('asset_id', np.array([f'obj_{k}' for k in range(nobj)]))
    assets=[text_value(v) for v in assets]
    # MOVi-C instance array indices are 0-based; segmentation values are index+1 (0 background).
    manifest=dict(scene_id=scene_id,data_origin=origin,
        source_video_id=text_value(ex['metadata'].get('video_name',scene_id)),
        fps=FPS,n_frames=T,duration_s=T/FPS,object_count=nobj,
        asset_ids=assets,segmentation_object_id_offset=1,
        full_mesh_included=False,pointcloud_coordinate_frame='camera frame at frame 0',
        speech_alignment='scene-level spoken question; NOT physically recorded in scene',
        impact_audio_origin='synthesized from MOVi-C collision events (not recorded acoustic audio)',
        tactile_origin='schematic force-conditioned contact patch (not physical tactile sensor)',
        n_collisions=int(len(coll['frame'])),
        n_valid_collisions=int(np.sum((coll['frame']>=0)&(coll['frame']<T)&(coll['force']>0))),
        primary_event_frame=qa.get('event_frame'),
        modalities={
            'rgb_video':'rgb.mp4','rgb_depth_seg_flow':'visual.npz',
            'motion_3d_boxes':'motion.npz','point_cloud':'pointcloud_camera_frame.npz',
            'collisions':'collisions.npz','force_tactile_proxy':'contact_tactile_proxy.npz',
            'impact_audio':'impact_synthetic.wav','speech_question':'spoken_question.wav','qa':'qa.json'
        })
    with open(out/'manifest.json','w') as f:json.dump(manifest,f,indent=2)
    return manifest

manifests=[]
for i,ex in enumerate(records):
    m=export_record(ex,i)
    manifests.append(m)
    print(f"[{i+1:02d}/{N_SCENES}] {m['scene_id']}: {m['object_count']} objects, "
          f"{m['n_collisions']} collision records, primary frame={m['primary_event_frame']}")
print("DONE. Exported",len(manifests),"scenes to",OUTPUT_ROOT)

[01/10] scene_0000: 9 objects, 2778 collision records, primary frame=6
[02/10] scene_0001: 9 objects, 3669 collision records, primary frame=8
[03/10] scene_0002: 8 objects, 2524 collision records, primary frame=8
[04/10] scene_0003: 6 objects, 2365 collision records, primary frame=9
[05/10] scene_0004: 6 objects, 2971 collision records, primary frame=7
[06/10] scene_0005: 10 objects, 5775 collision records, primary frame=6
[07/10] scene_0006: 5 objects, 1210 collision records, primary frame=9
[08/10] scene_0007: 8 objects, 2895 collision records, primary frame=6
[09/10] scene_0008: 4 objects, 1427 collision records, primary frame=8
[10/10] scene_0009: 7 objects, 2925 collision records, primary frame=7
DONE. Exported 10 scenes to /content/OneData_Omni_MOViC_10


## 5 · Automated alignment check (fail loudly)

Strict checks are based on event/frame identities, not on visually similar curves. We ensure frame lengths, valid event indices, force/tactile and synthesized impact WAV alignment, scene-level speech, point cloud data, and primary-event QA labels. **Passing checks means the exporter is internally consistent; it does not validate physical realism of synthetic tactile/audio.**

In [6]:
def check_scene(out):
    m=json.loads((out/'manifest.json').read_text())
    qa=json.loads((out/'qa.json').read_text())
    vis=np.load(out/'visual.npz'); mo=np.load(out/'motion.npz'); coll=np.load(out/'collisions.npz')
    ta=np.load(out/'contact_tactile_proxy.npz')
    pc=np.load(out/'pointcloud_camera_frame.npz')
    im,sr=sf.read(out/'impact_synthetic.wav')
    voice,voice_sr=sf.read(out/'spoken_question.wav')
    T=int(m['n_frames']); N=int(m['object_count'])
    checks={
      'RGB / depth / segmentation': vis['rgb'].shape==(T,128,128,3) and vis['depth_camera_distance'].shape==(T,128,128) and vis['segmentation'].shape==(T,128,128),
      '3D object trajectories': mo['positions_world'].shape==(N,T,3) and mo['velocities_world'].shape==(N,T,3),
      'frame timestamp':np.allclose(mo['frame_timestamp_s'],np.arange(T)/FPS,atol=1e-5),
      'contact/tactile tensor':ta['tactile_contact_patch'].shape==(T,TOUCH_SIZE,TOUCH_SIZE),
      'impact audio duration':sr==AUDIO_SR and len(im)==int(round(T/FPS*AUDIO_SR)),
      'speech WAV nonempty':voice_sr>0 and len(voice)>0 and np.max(np.abs(voice))>0,
      'point cloud produced':pc['xyz'].ndim==2 and pc['xyz'].shape[-1]==3 and len(pc['xyz'])>0,
      'primary event label':qa['event_frame'] is None or 0<=qa['event_frame']<T,
      'mp4 saved':(out/'rgb.mp4').exists() and (out/'rgb.mp4').stat().st_size>500,
    }
    f=np.asarray(coll['frame']); w=np.asarray(coll['force'])
    valid=np.flatnonzero((f>=0)&(f<T)&(w>0))
    if len(valid):
      active_frames=np.unique(f[valid])
      checks['force aligns at collision frames']=bool(np.all(ta['force_by_frame'][active_frames]>0))
      checks['tactile aligns at collision frames']=bool(np.all(ta['tactile_contact_patch'][active_frames].reshape(len(active_frames),-1).max(axis=1)>0))
      # The impact synth starts at the annotated video frame.
      checks['sound onset aligns at collision frames']=bool(all(
        np.max(np.abs(im[int(round(fr/FPS*sr)):int(round(fr/FPS*sr))+max(1,int(.01*sr))]))>1e-6
        for fr in active_frames))
      if qa['event_frame'] is not None:
        checks['QA references actual event']=int(qa['event_frame']) in set(active_frames.tolist())
    else:
      checks['no collision => no contact proxy']=bool(np.all(ta['force_by_frame']==0) and np.all(ta['tactile_contact_patch']==0))
    return checks

all_pass=True
for m in manifests:
    checks=check_scene(OUTPUT_ROOT/m['scene_id'])
    ok=all(checks.values());all_pass=all_pass and ok
    print(('PASS' if ok else 'FAIL'),m['scene_id'],f"({sum(checks.values())}/{len(checks)} checks)")
    if not ok:
        print({k:v for k,v in checks.items() if not v})
assert all_pass,'One or more alignment checks failed; inspect previous lines.'
print(f"✅ ALL {N_SCENES} SCENES PASSED alignment/export checks, source = {DATA_ORIGIN}")

PASS scene_0000 (13/13 checks)
PASS scene_0001 (13/13 checks)
PASS scene_0002 (13/13 checks)
PASS scene_0003 (13/13 checks)
PASS scene_0004 (13/13 checks)
PASS scene_0005 (13/13 checks)
PASS scene_0006 (13/13 checks)
PASS scene_0007 (13/13 checks)
PASS scene_0008 (13/13 checks)
PASS scene_0009 (13/13 checks)
✅ ALL 10 SCENES PASSED alignment/export checks, source = MOVi-C TFDS (google-research/kubric)


## 6 · Multimodal demo (real values; switch scenes with the dropdown)

This is the **final visual inspection**: same `frame` across the video, force trace and tactile image, plus native depth/segmentation, a 3D backprojected cloud and two generated WAV files. Choose a different scene and use the collision-frame slider.

Speech is a **question about the scene**, not a recorded sound from the animated world. Impact sound is synthesized and aligned to the collision timeline.

In [7]:
try:
    import ipywidgets as widgets
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
except ImportError:
    widgets = None
    print("ipywidgets not available: falling back to a noninteractive scene-0 demo.")

def show_scene(scene_idx=0,frame_choice=None):
    d=OUTPUT_ROOT/f'scene_{scene_idx:04d}'
    m=json.loads((d/'manifest.json').read_text())
    qa=json.loads((d/'qa.json').read_text())
    vis=np.load(d/'visual.npz');motion=np.load(d/'motion.npz');
    tc=np.load(d/'contact_tactile_proxy.npz')
    cloud=np.load(d/'pointcloud_camera_frame.npz')
    T=int(m['n_frames']);fr=int(frame_choice if frame_choice is not None else (m['primary_event_frame'] or T//2))
    fr=int(np.clip(fr,0,T-1));t=fr/FPS
    rgb=vis['rgb'][fr];seg=vis['segmentation'][fr];dep=vis['depth_camera_distance'][fr]
    tac=tc['tactile_contact_patch'][fr];force=tc['force_by_frame'];pos=motion['positions_world']
    fig=plt.figure(figsize=(16,10),facecolor='#f7f9fd')
    gs=fig.add_gridspec(2,3,wspace=.25,hspace=.24)
    ax=fig.add_subplot(gs[0,0]);ax.imshow(rgb);ax.set_title(f'RGB | frame {fr} / t={t:.2f}s');ax.axis('off')
    ax=fig.add_subplot(gs[0,1]);ax.imshow(seg,cmap='tab20',vmin=0,vmax=max(4,int(m['object_count'])+1));ax.set_title('Native instance segmentation');ax.axis('off')
    ax=fig.add_subplot(gs[0,2]); ax.imshow(dep,cmap='viridis');ax.set_title('Native depth (radial, scene units)');ax.axis('off')
    ax=fig.add_subplot(gs[1,0]);ax.imshow(tac,cmap='inferno',vmin=0,vmax=max(1e-4,float(np.max(tc['tactile_contact_patch']))));ax.set_title('Synthetic contact-patch tactile proxy');ax.axis('off')
    ax=fig.add_subplot(gs[1,1]);time=np.arange(T)/FPS
    ax.plot(time,force,color='#3762aa',linewidth=2);ax.axvline(t,color='orangered',ls='--');
    ax.set(xlabel='time (s)',ylabel='sum of annotated collision forces',title='Collision-aligned time series');ax.grid(alpha=.25)
    ax=fig.add_subplot(gs[1,2],projection='3d');pts=cloud['xyz'];clr=cloud['rgb']/255
    if len(pts):
       ix=np.linspace(0,len(pts)-1,min(1800,len(pts)),dtype=int)
       ax.scatter(pts[ix,0],pts[ix,1],pts[ix,2],c=clr[ix],s=2,depthshade=False)
    ax.set(xlabel='X',ylabel='Y',zlabel='Z');ax.set_title('RGB-D object point cloud (camera coords)')
    fig.suptitle(f'OneData-Omni | {m["scene_id"]} | {m["data_origin"]}',fontsize=15,fontweight='bold')
    fig.savefig(d/'demo_panel.png',dpi=115,bbox_inches='tight')
    plt.show()
    display(Markdown(f'**Spoken question:** {qa["question"]}  \n**Ground-truth answer:** {qa["answer"]}  \n**Collisions:** {m["n_collisions"]} | **Objects:** {m["object_count"]}'))
    display(Markdown('**Original MOVi-C RGB video**'))
    display(Video(str(d/'rgb.mp4'),embed=True,html_attributes='controls loop'))
    display(Markdown('**Synthetic event-aligned impact audio**'))
    display(Audio(filename=str(d/'impact_synthetic.wav')))
    display(Markdown('**Offline synthesized spoken question (scene-level)**'))
    display(Audio(filename=str(d/'spoken_question.wav')))

if widgets is not None:
    scene_picker=widgets.Dropdown(options=[(m['scene_id'],i) for i,m in enumerate(manifests)],value=0,description='Scene:')
    frame_slider=widgets.IntSlider(value=12,min=0,max=23,step=1,description='Frame:',continuous_update=False)
    demo_out=widgets.Output()
    def update_demo(change=None):
        with demo_out:
            demo_out.clear_output(wait=True)
            show_scene(scene_picker.value,frame_slider.value)
    scene_picker.observe(update_demo,names='value')
    frame_slider.observe(update_demo,names='value')
    display(widgets.HBox([scene_picker,frame_slider]),demo_out)
    update_demo()
else:
    show_scene(0)


Output()

## 7 · Export downloadable dataset ZIP + summary

Use this ZIP with your teammates. Each scene includes explicit modality labels and provenance. If you selected `mock_selftest`, the archive is labeled accordingly; *do not* treat it as real MOVi-C training data. The archive includes generated videos and audio so it can be inspected without TensorFlow.

In [8]:
from zipfile import ZipFile, ZIP_DEFLATED
zip_path=OUTPUT_ROOT.parent/("OneData_Omni_MOViC_10.zip" if DATA_SOURCE=="movi_c" else "OneData_Omni_MOCK_ONLY_10.zip")
with ZipFile(zip_path,'w',compression=ZIP_DEFLATED,compresslevel=3) as zf:
    for p in sorted(OUTPUT_ROOT.rglob('*')):
        if p.is_file(): zf.write(p,arcname=str(p.relative_to(OUTPUT_ROOT.parent)))
print("ZIP ready:",zip_path, "size MB:",round(zip_path.stat().st_size/1e6,1))
print("Origin:",DATA_ORIGIN)
print("Scene count:",len(manifests))
print("Total annotated collision records:",sum(m['n_collisions'] for m in manifests))
try:
    from google.colab import files
    # Uncomment when you want to transfer the archive to your machine:
    # files.download(str(zip_path))
except ImportError: pass

ZIP ready: /content/OneData_Omni_MOViC_10.zip size MB: 51.1
Origin: MOVi-C TFDS (google-research/kubric)
Scene count: 10
Total annotated collision records: 28539


## Notes for the next iteration

- Native **GSO Mesh** assets require downloading the appropriate original Google Scanned Objects geometry separately, using each scene's `asset_ids`; the current TFDS source provides native 3D state and a derived RGB-D point cloud, **not full meshes**.
- One MOVi-C video uses a particular camera; RGB across time is **not** guaranteed multi-view coverage. To create true independently rendered 3-view images, obtain meshes and rerender from new cameras.
- Impact WAV and tactile proxy preserve **event timestamps**, but are not ground-truth acoustics/tactile physics. Later replace them with [ObjectFolder](https://github.com/rhgao/ObjectFolder) or [TACTO](https://github.com/facebookresearch/tacto) if you need realism.
- Per-scene spoken question is automatically rendered with offline `espeak-ng`; it is **scene-level semantically aligned** but not a synchronous voice captured during the video.
- No LLM training is performed in this notebook. It only validates the intended **data generation and alignment** stages.